# Debugging a NaN test loss

After training, the **test loss came out as NaN** even though the training loss looked normal. This notebook is the investigation that traced it back to **float16 overflow** in the first decoder block.

To run these cells, first run sections 1–5 of the main notebook (`unet_landcover_segmentation.ipynb`) up to and including loading the checkpoint, so `model` and `checkpoint` exist.

Most cell outputs from the original broken run weren't kept, because the checkpoint was repaired afterwards.

**Summary:** under mixed precision the convolutions run in float16, which can't store numbers above 65,504. `up1`'s first convolution adds up 9,216 inputs per output pixel, so its output overflowed to inf. That inf then corrupted the BatchNorm layer's running statistics, which are only used in eval mode, so only validation/test broke. The fix is to run `up1` and `up2` in float32. The full explanation is at the end.

## Step 1: Which saved parameters contain NaN?

In [ ]:
# Looking through every saved parameter and keeping the names of any that contain NaN
state = checkpoint["model_state_dict"]

nan_parameters = []

for name, parameter in state.items():
    if torch.isnan(parameter).any():
        nan_parameters.append(name)

# Each name reads like a path through the model, ending where the NaN is
print("Parameters containing NaN:\n")
for param in nan_parameters:
  print(param)

In [ ]:
#checking for infinity values

inf_parameters = []

for name, parameter in state.items():

    if torch.isinf(parameter).any():
        inf_parameters.append(name)

print("Parameters containing Inf:")
print(inf_parameters)

## Step 2: Check the BatchNorm layers

BatchNorm keeps a `running_mean` and `running_var` during training, and uses them in eval mode. If those are corrupted, only eval/test breaks, which matches what was happening.

In [ ]:
#After reading my output from the above cell I know the location of the nan value so I want to now trace it back to find where it was created so I'll look through the layers

for name, module in model.named_modules():
  if isinstance(module, torch.nn.BatchNorm2d):
    print(name)

_orig_mod.inc.conv.1
_orig_mod.inc.conv.4
_orig_mod.down1.down.1.conv.1
_orig_mod.down1.down.1.conv.4
_orig_mod.down2.down.1.conv.1
_orig_mod.down2.down.1.conv.4
_orig_mod.down3.down.1.conv.1
_orig_mod.down3.down.1.conv.4
_orig_mod.down4.down.1.conv.1
_orig_mod.down4.down.1.conv.4
_orig_mod.up1.conv.conv.1
_orig_mod.up1.conv.conv.4
_orig_mod.up2.conv.conv.1
_orig_mod.up2.conv.conv.4
_orig_mod.up3.conv.conv.1
_orig_mod.up3.conv.conv.4
_orig_mod.up4.conv.conv.1
_orig_mod.up4.conv.conv.4


In [ ]:
# For each BatchNorm layer, check whether its running mean or running variance contains NaN
for name, module in model.named_modules():
    if isinstance(module, torch.nn.BatchNorm2d):
        print(
            name,
            "mean NaN:", torch.isnan(module.running_mean).any().item(),
            "var NaN:", torch.isnan(module.running_var).any().item()
        )

The NaN first appears in `up1.conv.conv.1`, the BatchNorm right after `up1`'s first convolution. A closer look at that layer:

In [ ]:
bn = model.up1.conv.conv[1]

nan_positions = torch.isnan(bn.running_mean)

print("Number of NaN means:", nan_positions.sum().item())
print("Total channels:", bn.running_mean.numel())

In [ ]:
nan_positions = torch.isnan(bn.running_var)

print("Number of NaN variances:", nan_positions.sum().item())
print("Total channels:", bn.running_var.numel())

In [ ]:
# The learnable BatchNorm weights are fine, so only the running stats are corrupted

print("weight nan:", torch.isnan(bn.weight).any().item())
print("weight inf:", torch.isinf(bn.weight).any().item())

print("bias nan:", torch.isnan(bn.bias).any().item())
print("bias inf:", torch.isinf(bn.bias).any().item())

## Step 3: Find the first layer that outputs NaN/Inf

A **forward hook** is a function that runs every time a layer produces an output. One is put on every layer to report the first one whose output isn't finite, along with its input. `torch.compile` stops hooks from firing, so they go on the uncompiled model.

In [ ]:
# Remove any hooks left over from earlier runs
for m in model.modules():
    if hasattr(m, '_forward_hooks'):
        m._forward_hooks.clear()
    if hasattr(m, '_forward_pre_hooks'):
        m._forward_pre_hooks.clear()
    if hasattr(m, '_backward_hooks'):
        m._backward_hooks.clear()

# Unwrap the compiled model so the hooks fire (only if it's still compiled, so the cell can be re-run)
if hasattr(model, '_orig_mod'):
  model = model._orig_mod


def check_module_output(name):

  def hook(module, input, output):

    if isinstance(output, torch.Tensor) and not torch.isfinite(output).all():

      print("\n🚨 BAD MODULE:", name)

      # Input
      if isinstance(input[0], torch.Tensor):
        print("Input finite:", torch.isfinite(input[0]).all().item())
        print("Input max:", input[0].abs().max().item())
        print("Input min:", input[0].abs().min().item())

      # Output
      print("Output finite:", torch.isfinite(output).all().item())
      print("Output max:", output.abs().max().item())
      print("Output min:", output.abs().min().item())

      raise RuntimeError(f"\n🚨 NaN/Inf detected in module: {name}")

  return hook


hooks = []

for name, module in model.named_modules():
  hooks.append(module.register_forward_hook(check_module_output(name)))

The hook flagged `up1.conv.conv.0` with a **finite input but an inf output**, so the overflow happens inside that layer. Checking its weights rules them out:

In [ ]:
model.up1.conv.conv[0]

Conv2d(1024, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)

In [ ]:
layer = model.up1.conv.conv[0]

print(layer)
print("weight dtype:", layer.weight.dtype)
print("weight finite:", torch.isfinite(layer.weight).all().item())
print("weight max:", layer.weight.abs().max().item())
print("weight min:", layer.weight.abs().min().item())

## Step 4: Confirming float16 is involved

Inside `torch.autocast` on a GPU, convolutions output float16:

In [ ]:
conv = torch.nn.Conv2d(3, 64, kernel_size=3).cuda()

x = torch.randn(8, 3, 256, 256, device="cuda")

with torch.autocast(device_type="cuda"):
    y = conv(x)

print(y.dtype)

torch.float16


## What caused it, and the fix

The weights were never the problem. They're float32, finite, and the biggest is 0.42. The min of 1.26e-08 just means one of the 4.7 million weights is almost exactly 0, which is normal.

The problem is float16. Training runs inside torch.autocast, so on the T4 the convs run in float16, and the biggest number float16 can store is 65,504. Anything bigger turns into inf.

up1.conv.conv.0 is Conv2d(1024, 512, kernel_size=3). For every output pixel it adds up 1024 × 3 × 3 = 9,216 weighted inputs, making it one of the two widest convs in the model. Half of its input comes straight from the ConvTranspose2d, which has no BatchNorm keeping its values small. The hook showed its input reached 979. With 9,216 weighted inputs, each one only needs to be about 7.1 on average (65,504 ÷ 9,216) for the total to overflow. Input finite + output inf = the overflow happened inside this layer.

When that inf reaches the next layer (BatchNorm up1.conv.conv.1) in train mode, its running_mean becomes inf and running_var becomes NaN, and they never recover. That's the NaN that showed up in running_mean/running_var earlier. It only breaks eval/test because only eval mode uses the running stats. Gradient clipping couldn't stop it because the inf happens in the forward pass, before any gradients exist.

The fix is in the UNet cell: up1 and up2 run with autocast turned off, so they use float32, which goes up to about 3.4 × 10^38. Everything else still gets the float16 speed-up. No layers were added, so the saved checkpoint still loads.

If a hook ever reports another module with "Input finite: True" and "Output finite: False", it's the same overflow, and moving that block inside the same `with torch.autocast(..., enabled=False):` (with .float() on its inputs) fixes it.

## Repairing the corrupted checkpoint

Before the root cause was found, the corrupted running stats were repaired by resetting them and recalculating them with forward passes over the training data (no weight updates). Kept here for reference; it isn't needed once up1/up2 run in float32.

In [ ]:
# import tqdm

# #The actual fix for my corrupted var and mean my reseting them

# for module in model.modules():
#   if isinstance(module, torch.nn.BatchNorm2d):
#     module.reset_running_stats()
#     module.momentum = None


# #going to train my model only to restore the running stats not to change the params
# model.train()

# with torch.no_grad():
#     for image, mask in tqdm.tqdm(
#         train_dataloader,
#         desc="Repairing running stats"
#     ):
#         image = image.to(device)
#         output = model(image)

# model.eval()

In [ ]:
# import torch

# for name, module in model.named_modules():

#   if isinstance(module, torch.nn.BatchNorm2d):


#       print("NAN total(running_mean):", torch.isnan(module.running_mean).sum().item())

#       print("NAN total(running_var):", torch.isnan(module.running_var).sum().item())

#       print("inf total(running_mean):", torch.isinf(module.running_mean).sum().item())

#       print("inf total(running_var):", torch.isinf(module.running_var).sum().item())

In [ ]:
# model.eval()

# with torch.inference_mode():
#     for batch_number, (image, mask) in enumerate(test_dataloader, start=1):

#         image = image.to(device)
#         mask = mask.to(device)

#         output = model(image)

#         print(
#             f"Batch {batch_number}:",
#             "NaN =", torch.isnan(output).any().item(),
#             "Inf =", torch.isinf(output).any().item()
#         )

## Checking the fix

In [ ]:
image, mask = next(iter(test_dataloader))

image = image.to(device)
mask = mask.to(device)

model.eval()

with torch.inference_mode():
    test_output = model(image)

pred_mask = torch.argmax(test_output, dim=1)

print("Output shape:", test_output.shape)
print("Output NaN:", torch.isnan(test_output).any().item())
print("Output Inf:", torch.isinf(test_output).any().item())

print("Mask dtype:", mask.dtype)
print("Mask shape:", mask.shape)
print("Mask unique values:", torch.unique(mask))

print("Predicted classes:", torch.unique(pred_mask))

Output shape: torch.Size([4, 7, 512, 512])
Output NaN: False
Output Inf: False
Mask dtype: torch.int64
Mask shape: torch.Size([4, 512, 512])
Mask unique values: tensor([0, 2, 3, 5], device='cuda:0')
Predicted classes: tensor([0, 1, 2, 3, 4, 5], device='cuda:0')


## Other fixes made along the way

- `random_split` uses a fixed seed, and the training dataset is no longer re-created after the split, so validation images aren't in the training set.
- An inf gradient now lets `GradScaler` skip the step instead of raising an error.
- A checkpoint isn't saved if the validation loss is NaN/inf, so a broken model can't overwrite a good one.